In [ ]:
# CELL 1 — vickymacm2 configuration and target preflight. The environment may override INPUT_ROOT.
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import runpy

INPUT_ROOT = Path(
    os.environ.get(
        "KAGGLE_VLLM_INPUT_ROOT",
        "/kaggle/input/datasets/vickymacm2",
    )
)
CANDIDATE_DATASET_ROOT = INPUT_ROOT / "kaggle-vllm-v030-production-acceptance"
NATIVE_DATASET_ROOT = INPUT_ROOT / "kaggle-vllm-v030-native-wheel-only"
NATIVE_DATASET_WHEEL = NATIVE_DATASET_ROOT / "vllm-0.30.0-cp312-cp312-linux_x86_64.whl"
if NATIVE_DATASET_ROOT.is_dir():
    assert NATIVE_DATASET_WHEEL.is_file(), f"Native Dataset is attached but exact wheel is missing: {NATIVE_DATASET_WHEEL}"
    OPTIONAL_NATIVE_WHEEL = NATIVE_DATASET_WHEEL
else:
    OPTIONAL_NATIVE_WHEEL = None  # Deterministic P31 saved Notebook Output fallback in the runner.
CLEANUP_RUNTIME_AFTER_PASS = False  # Optional; retains /tmp runtime for debugging by default.
RUNTIME_ROOT = Path("/tmp/kaggle-vllm/v0.3.0/l4x4")
EVIDENCE_ROOT = Path("/kaggle/working/kaggle-vllm-v030-production-acceptance")

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12), "Requires Kaggle CPython 3.12"
print("Python:", sys.executable, sys.version)
for command in (["nproc", "--all"], ["nvidia-smi", "-L"], ["df", "-hT", "/kaggle/working", "/tmp"]):
    result = subprocess.run(command, capture_output=True, text=True, timeout=30, check=True)
    print("$", " ".join(command), "\n" + result.stdout)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], capture_output=True, text=True, timeout=30, check=True)
rows = [tuple(x.strip() for x in line.split(",", 1)) for line in gpu.stdout.splitlines() if line.strip()]
assert rows == [("NVIDIA L4", "8.9")] * 4, f"Requires exactly four L4/SM89 GPUs: {rows}"
free = shutil.disk_usage("/tmp").free
assert free >= 24 * 1024**3, f"/tmp needs at least 24 GiB free; found {free} bytes"
assert CANDIDATE_DATASET_ROOT.is_dir(), f"Attach the final-candidate Dataset at {CANDIDATE_DATASET_ROOT}"
assert RUNTIME_ROOT == Path("/tmp/kaggle-vllm/v0.3.0/l4x4")


In [ ]:
# CELL 2 — verify the exact small candidate Dataset before running any child process.
checksum_file = CANDIDATE_DATASET_ROOT / "SHA256SUMS.txt"
assert checksum_file.is_file(), f"Missing {checksum_file}"
verified = {}
for line in checksum_file.read_text().splitlines():
    expected, separator, relative = line.partition("  ")
    assert separator and len(expected) == 64 and relative, f"Invalid checksum entry: {line!r}"
    item = CANDIDATE_DATASET_ROOT / relative
    assert not item.is_symlink() and item.is_file() and item.resolve().parent == CANDIDATE_DATASET_ROOT.resolve(), f"Unsafe or missing item: {relative}"
    digest = hashlib.sha256()
    with item.open("rb") as source:
        for block in iter(lambda: source.read(4 * 1024 * 1024), b""):
            digest.update(block)
    assert digest.hexdigest() == expected, f"Candidate checksum mismatch: {relative}"
    verified[relative] = expected
SDK_WHEEL = CANDIDATE_DATASET_ROOT / "kaggle_vllm-0.3.0-py3-none-any.whl"
RUNNER = CANDIDATE_DATASET_ROOT / "production_acceptance_v030.py"
assert verified.get(SDK_WHEEL.name) == "7da66878a85c3b65db18d794d282519d853893f54d7314db5ae551ace377416b"
assert RUNNER.name in verified, "Runner must be inside the verified final-candidate Dataset"
assert len(verified) >= 10, "Candidate Dataset is incomplete"
print(f"Verified {len(verified)} small candidate files in {CANDIDATE_DATASET_ROOT}")


In [ ]:
# CELL 3 — run the streamed, timeout-bound acceptance gates with explicit inputs.
arguments = ["--input-root", str(INPUT_ROOT), "--sdk-wheel", str(SDK_WHEEL)]
if OPTIONAL_NATIVE_WHEEL is not None:
    arguments.extend(["--native-wheel", str(Path(OPTIONAL_NATIVE_WHEEL))])
if CLEANUP_RUNTIME_AFTER_PASS:
    arguments.append("--cleanup-runtime-after-pass")
print("Runner:", RUNNER, "arguments:", arguments, flush=True)
exit_code = runpy.run_path(str(RUNNER))["main"](arguments)
assert exit_code == 0, f"Fresh SDK acceptance failed with exit code {exit_code}; inspect {EVIDENCE_ROOT}"


In [ ]:
# CELL 4 — show the final matrix and exact downloadable evidence ZIP identity.
matrix_file = EVIDENCE_ROOT / "evidence/19_final_matrix.json"
bundle_file = EVIDENCE_ROOT / "20_evidence_bundle.zip"
assert matrix_file.is_file() and bundle_file.is_file(), "Acceptance evidence is incomplete"
matrix = json.loads(matrix_file.read_text())
print(json.dumps(matrix, indent=2))
assert matrix["status"] == "FRESH_KAGGLE_PASS", matrix["status"]
bundle_sha = hashlib.sha256()
with bundle_file.open("rb") as source:
    for block in iter(lambda: source.read(4 * 1024 * 1024), b""):
        bundle_sha.update(block)
print("Evidence ZIP SHA256:", bundle_sha.hexdigest())
print("Download:", bundle_file)
